# 04 · Data warehouse y OLAP — constelación de hechos

Dos tablas de hechos (`hechos_entregas`, `hechos_incidentes`) comparten las dimensiones
`dim_tiempo`, `dim_geografia` y `dim_vehiculo` → **Fact Constellation**.

```
            dim_tiempo ─────┬───────────────┐
                            │               │
 dim_vehiculo ── hechos_entregas      hechos_incidentes ── dim_vehiculo
                  │       │                   │
        dim_geografia (origen/destino)   dim_geografia
```

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from logistica import etl
from logistica.consultas import consultar

pd.set_option("display.width", 160)
print(etl.ejecutar_etl())

## Slice — se fija una dimensión: año 2026

In [ ]:
display(
    consultar(
        "dw",
        """
    SELECT g.region, count(*) AS entregas, round(avg(h.horas_entrega), 2) AS horas_promedio
    FROM hechos_entregas h
    JOIN dim_tiempo t ON t.tiempo_id = h.tiempo_id
    JOIN dim_geografia g ON g.geografia_id = h.geografia_origen_id
    WHERE t.anio = 2026
    GROUP BY g.region ORDER BY g.region""",
    )
)

## Dice — sub-cubo: regiones Norte y Sur, camiones refrigerados, por trimestre

In [ ]:
display(
    consultar(
        "dw",
        """
    SELECT t.anio, t.trimestre, g.region, v.tipo, count(*) AS entregas,
           round(avg(h.horas_entrega), 2) AS horas_promedio
    FROM hechos_entregas h
    JOIN dim_tiempo t ON t.tiempo_id = h.tiempo_id
    JOIN dim_geografia g ON g.geografia_id = h.geografia_origen_id
    JOIN dim_vehiculo v ON v.vehiculo_id = h.vehiculo_id
    WHERE g.region IN ('norte', 'sur') AND v.tipo = 'camion_refrigerado'
    GROUP BY t.anio, t.trimestre, g.region, v.tipo
    ORDER BY t.anio, t.trimestre, g.region""",
    )
)

## Roll-up — de ciudad a provincia, región y total (`ROLLUP`)

In [ ]:
rollup = consultar(
    "dw",
    """
    SELECT g.region, g.provincia, g.ciudad, count(*) AS entregas,
           round(avg(h.horas_entrega), 1) AS horas_promedio
    FROM hechos_entregas h JOIN dim_geografia g ON g.geografia_id = h.geografia_origen_id
    GROUP BY ROLLUP (g.region, g.provincia, g.ciudad)
    ORDER BY g.region NULLS LAST, g.provincia NULLS LAST, g.ciudad NULLS LAST""",
)
display(rollup.head(12))

## Cubo — todas las combinaciones de región × tipo de vehículo × año (`CUBE`)

In [ ]:
display(
    consultar(
        "dw",
        """
    SELECT g.region, v.tipo AS vehiculo, t.anio, count(*) AS entregas,
           round(avg(h.horas_entrega), 1) AS horas_promedio
    FROM hechos_entregas h
    JOIN dim_tiempo t ON t.tiempo_id = h.tiempo_id
    JOIN dim_geografia g ON g.geografia_id = h.geografia_origen_id
    JOIN dim_vehiculo v ON v.vehiculo_id = h.vehiculo_id
    GROUP BY CUBE (g.region, v.tipo, t.anio)
    ORDER BY g.region NULLS LAST, v.tipo NULLS LAST, t.anio NULLS LAST""",
    ).head(15)
)

## Pregunta de negocio: tiempo promedio de entrega por región, año actual vs. anterior

In [ ]:
yoy = consultar(
    "dw",
    """
    SELECT g.region,
           round(avg(h.horas_entrega) FILTER (WHERE t.anio = 2025), 2) AS horas_2025,
           round(avg(h.horas_entrega) FILTER (WHERE t.anio = 2026), 2) AS horas_2026
    FROM hechos_entregas h
    JOIN dim_tiempo t ON t.tiempo_id = h.tiempo_id
    JOIN dim_geografia g ON g.geografia_id = h.geografia_origen_id
    GROUP BY g.region ORDER BY g.region""",
)
yoy["variacion_pct"] = ((yoy.horas_2026 / yoy.horas_2025 - 1) * 100).round(1)
display(yoy)

ax = yoy.set_index("region")[["horas_2025", "horas_2026"]].plot.bar(figsize=(7, 3.5), rot=0)
ax.set_ylabel("horas promedio de entrega")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

La región Sur muestra la mejora más fuerte respecto de 2025 (efecto sembrado en los datos sintéticos para
que la pregunta tenga respuesta; el resto varía por azar); con datos reales sería la señal a investigar.

## Drill-across — combinar ambas tablas de hechos por dimensiones compartidas

Los incidentes por mes y región junto al tiempo de entrega de esa región y mes: solo es posible porque
las dos tablas de hechos usan las mismas dimensiones (constelación).

In [ ]:
cruce = consultar(
    "dw",
    """
    WITH entregas AS (
        SELECT g.region, t.anio, t.trimestre, round(avg(h.horas_entrega), 1) AS horas_promedio
        FROM hechos_entregas h
        JOIN dim_tiempo t ON t.tiempo_id = h.tiempo_id
        JOIN dim_geografia g ON g.geografia_id = h.geografia_origen_id
        GROUP BY 1, 2, 3
    ), incidentes AS (
        SELECT g.region, t.anio, t.trimestre, count(*) AS incidentes,
               round(avg(i.demora_min)) AS demora_media_min
        FROM hechos_incidentes i
        JOIN dim_tiempo t ON t.tiempo_id = i.tiempo_id
        JOIN dim_geografia g ON g.geografia_id = i.geografia_id
        GROUP BY 1, 2, 3
    )
    SELECT * FROM entregas JOIN incidentes USING (region, anio, trimestre)
    ORDER BY region, anio, trimestre""",
)
display(cruce.head(12))

## Particionamiento por rango de las tablas de hechos

`hechos_entregas` y `hechos_incidentes` se particionan por `RANGE (tiempo_id)`, una partición por año. Una
consulta de un solo año toca solo su partición (*partition pruning*).

In [ ]:
from logistica.conexiones import conectar_pg

with conectar_pg("dw") as c:
    for f in c.execute(
        "EXPLAIN SELECT count(*) FROM hechos_entregas WHERE tiempo_id BETWEEN 20260101 AND 20261231"
    ):
        print(f["QUERY PLAN"])